# Data Analytics — Train the Trainer
## Student Copy

**10 Days · 80 Hours · Google Colaboratory**
Aligned to the TESDA Competency Standards for Data Analytics Level III.

---
### How each day is structured

| Time | Segment |
|---|---|
| 8:00 – 9:00 | Lecture 1 |
| 9:00 – 10:00 | Hands-On 1 |
| 10:00 – 10:10 | Break |
| 10:10 – 10:40 | Analysis and Discussion 1 |
| 10:40 – 11:40 | Lecture 2 |
| 11:40 – 12:40 | Hands-On 2 |
| 12:40 – 1:40 | Lunch |
| 1:40 – 2:10 | Analysis and Discussion 2 |
| 2:10 – 3:10 | Lecture 3 |
| 3:10 – 4:10 | Hands-On 3 |
| 4:10 – 4:20 | Break |
| 4:20 – 4:50 | Analysis and Discussion 3 |
| 4:50 – 5:00 | Wrap-Up |

Fill in the cells marked `# your code here`. Your instructor has the answer key.

---
*Prepared by Nina Comia for the Department of Information and Communications Technology (DICT).*

## Setup — Run This First (Every Session)
Colab clears files when the runtime disconnects. If your variables vanish, run this cell again.

In [ ]:
# ============================================================
# SETUP — run this cell first, every session.
# Creates the two course data files inside ./data/
#   service_requests.csv   (1,222 rows, with deliberate defects)
#   regional_offices.json  (9 offices, nested structure)
# ============================================================
import pandas as pd, numpy as np, json, random, os

np.random.seed(2026); random.seed(2026)
os.makedirs("data", exist_ok=True)

REGIONS = ["NCR", "Region III", "Region IV-A", "Region VI", "Region VII", "Region XI"]
OFFICES = {"NCR": ["DICT-NCR-01", "DICT-NCR-02"], "Region III": ["DICT-R3-01"],
           "Region IV-A": ["DICT-R4A-01", "DICT-R4A-02"], "Region VI": ["DICT-R6-01"],
           "Region VII": ["DICT-R7-01", "DICT-R7-02"], "Region XI": ["DICT-R11-01"]}
SERVICES = ["Free WiFi Installation", "Digital Literacy Training", "Business Permit Assistance",
            "ICT Equipment Request", "Cybersecurity Advisory", "eGov App Support"]
CHANNELS = ["Walk-in", "Online Portal", "Email", "Hotline", "Mobile App"]
STATUS = ["Resolved", "Pending", "Escalated", "Closed - No Action"]
AGE_GROUPS = ["18-24", "25-34", "35-44", "45-54", "55+"]

dates = pd.date_range("2025-01-01", "2025-12-31", freq="D")
rows = []
for i in range(1, 1201):
    region = random.choice(REGIONS)
    status = random.choices(STATUS, weights=[62, 20, 12, 6])[0]
    rows.append({
        "request_id": f"SR-2025-{i:05d}",
        "date_filed": random.choice(dates),
        "region": region,
        "office_code": random.choice(OFFICES[region]),
        "service_type": random.choice(SERVICES),
        "channel": random.choices(CHANNELS, weights=[30, 25, 15, 15, 15])[0],
        "status": status,
        "days_to_resolve": max(0, int(np.random.lognormal(1.6, 0.75))) if status in ("Resolved", "Closed - No Action") else np.nan,
        "satisfaction_rating": random.choice([1,2,3,4,5]) if status == "Resolved" else np.nan,
        "citizen_age_group": random.choice(AGE_GROUPS),
        "processing_fee": float(random.choice([0, 0, 0, 50, 100, 150, 200])),
    })
df = pd.DataFrame(rows)

# Inject realistic data-quality defects (used in the Day 5 cleaning lessons)
d = df.copy()
_ncr = d.index[d["region"] == "NCR"]
d.loc[pd.Index(np.random.RandomState(1).choice(_ncr, 40, replace=False)), "region"] = " ncr "
d.loc[d.sample(25, random_state=2).index, "service_type"] = "Free Wifi Installation"
d.loc[d.sample(30, random_state=3).index, "citizen_age_group"] = np.nan
d.loc[d.sample(18, random_state=4).index, "processing_fee"] = np.nan
d.loc[d.sample(8,  random_state=5).index, "days_to_resolve"] = 999
d = pd.concat([d, d.sample(22, random_state=6)])
d["date_filed"] = d["date_filed"].astype(str)
_mix = d.sample(60, random_state=7).index
d.loc[_mix, "date_filed"] = pd.to_datetime(d.loc[_mix, "date_filed"]).dt.strftime("%m/%d/%Y")
d = d.sample(frac=1, random_state=8).reset_index(drop=True)
d.to_csv("data/service_requests.csv", index=False)

CITIES = {"NCR": "Quezon City", "Region III": "San Fernando", "Region IV-A": "Calamba",
          "Region VI": "Iloilo City", "Region VII": "Cebu City", "Region XI": "Davao City"}
offices = []
for region, codes in OFFICES.items():
    for c_ in codes:
        offices.append({
            "office_code": c_,
            "office_name": f"DICT Field Office {c_.split('-')[-1]} - {region}",
            "location": {"region": region, "city": CITIES[region],
                         "coordinates": {"lat": round(random.uniform(6.0, 16.5), 4),
                                         "lon": round(random.uniform(120.0, 126.0), 4)}},
            "staff_count": random.randint(8, 45),
            "year_established": random.randint(2012, 2022),
            "annual_budget_php": random.randrange(2_000_000, 12_000_000, 250_000),
            "services_offered": random.sample(SERVICES, k=random.randint(3, 6)),
            "is_active": random.random() > 0.1,
        })
with open("data/regional_offices.json", "w") as f:
    json.dump({"agency": "Department of Information and Communications Technology",
               "dataset": "Regional Field Office Master Data", "as_of": "2025-12-31",
               "record_count": len(offices), "offices": offices}, f, indent=2)

print("Setup complete. Files in ./data/:")
for fn in sorted(os.listdir("data")):
    print("   ", fn)
print("\nCSV rows:", len(d), "| Offices:", len(offices))

Setup complete. Files in ./data/:
    regional_offices.json
    service_requests.csv
    service_requests_clean_v1.csv

CSV rows: 1222 | Offices: 9


---
# Day 7 — Summarizing Data Sets: Aggregation and Pivot Analysis
*TESDA alignment: Summarize data sets*

## Lecture 1 — Split, Apply, Combine

Aggregation follows one pattern: **split** the rows into groups, **apply** a calculation to each group, **combine** the results into a table.

```
df.groupby("region")["days_to_resolve"].mean()
```

Split by region, apply mean, combine into one row per region.

`.agg()` runs several functions at once and names the outputs:

```
df.groupby("region").agg(
    requests=("request_id", "count"),
    avg_days=("days_to_resolve", "mean"))
```

Group by more than one column by passing a list. The result gets a multi-level index, which `.reset_index()` flattens back into ordinary columns.

The equivalent in Excel is SUMIFS and COUNTIFS. The equivalent in SQL is GROUP BY. Same idea, three syntaxes.

In [ ]:
import pandas as pd, numpy as np
raw = pd.read_csv("data/service_requests.csv")
clean = raw.drop_duplicates().copy()
clean["region"] = clean["region"].str.strip().str.upper()
clean["service_type"] = clean["service_type"].replace({"Free Wifi Installation": "Free WiFi Installation"})
clean["date_filed"] = pd.to_datetime(clean["date_filed"], format="mixed")

# 1
q1, q3 = clean["days_to_resolve"].quantile([0.25, 0.75])
upper_fence = q3 + 1.5 * (q3 - q1)
print(f"Upper fence: {upper_fence:.1f}")

# 2
outlier_count = (clean["days_to_resolve"] > upper_fence).sum()
sentinel_count = (clean["days_to_resolve"] == 999).sum()
print("Above fence:", outlier_count, "| Exactly 999:", sentinel_count)

# 3. Replace sentinels — they are placeholders, not measurements
clean.loc[clean["days_to_resolve"] == 999, "days_to_resolve"] = np.nan

# 4. Integrity checks
resolved = clean[clean["status"] == "Resolved"]
integrity = {
    "resolved_have_days": bool(resolved["days_to_resolve"].notna().all()),
    "dates_in_2025": bool(clean["date_filed"].dt.year.eq(2025).all()),
    "no_dup_ids": bool(~clean["request_id"].duplicated().any()),
}
for k, v in integrity.items():
    print(f"{'PASS' if v else 'FAIL'}  {k}")

# 5. Save versioned output — raw file stays untouched
clean.to_csv("data/service_requests_clean_v1.csv", index=False)
print("\nSaved data/service_requests_clean_v1.csv with", len(clean), "rows")

Upper fence: 18.0
Above fence: 41 | Exactly 999: 8
FAIL  resolved_have_days
PASS  dates_in_2025
PASS  no_dup_ids

Saved data/service_requests_clean_v1.csv with 1200 rows


In [ ]:
import pandas as pd
clean = pd.read_csv("data/service_requests_clean_v1.csv")
clean.head()

,request_id,date_filed,region,office_code,service_type,channel,status,days_to_resolve,satisfaction_rating,citizen_age_group,processing_fee
0,SR-2025-01007,2025-12-17,REGION III,DICT-R3-01,ICT Equipment Request,Hotline,Resolved,4.0,4.0,55+,0.0
1,SR-2025-00391,2025-03-01,REGION III,DICT-R3-01,Cybersecurity Advisory,Walk-in,Pending,NaN,NaN,35-44,0.0
2,SR-2025-00794,2025-01-12,REGION VI,DICT-R6-01,Free WiFi Installation,Online Portal,Pending,NaN,NaN,55+,50.0
3,SR-2025-00147,2025-05-08,REGION XI,DICT-R11-01,Free WiFi Installation,Online Portal,Resolved,5.0,2.0,18-24,200.0
4,SR-2025-00387,2025-07-23,REGION VII,DICT-R7-01,Cybersecurity Advisory,Hotline,Resolved,2.0,3.0,25-34,0.0


In [ ]:
clean.groupby("region")["days_to_resolve"].mean()

,days_to_resolve
region,
NCR,6.139706
REGION III,6.532847
REGION IV-A,7.758389
REGION VI,6.171642
REGION VII,5.684211
REGION XI,6.450382


In [ ]:
clean.groupby("region")["days_to_resolve"].agg(["count", "mean", "max"])

,count,mean,max
region,,,
NCR,136,6.139706,41.0
REGION III,137,6.532847,45.0
REGION IV-A,149,7.758389,38.0
REGION VI,134,6.171642,30.0
REGION VII,114,5.684211,20.0
REGION XI,131,6.450382,38.0


In [ ]:
clean.groupby("region").agg(request=("request_id", "max"),avg_days=("days_to_resolve", "mean"),)

,request,avg_days
region,,
NCR,SR-2025-01197,6.139706
REGION III,SR-2025-01199,6.532847
REGION IV-A,SR-2025-01195,7.758389
REGION VI,SR-2025-01177,6.171642
REGION VII,SR-2025-01196,5.684211
REGION XI,SR-2025-01200,6.450382


In [ ]:
clean.groupby("region").agg(request=("request_id","max"))

,request
region,
NCR,SR-2025-01197
REGION III,SR-2025-01199
REGION IV-A,SR-2025-01195
REGION VI,SR-2025-01177
REGION VII,SR-2025-01196
REGION XI,SR-2025-01200


In [ ]:
import pandas as pd
clean = pd.read_csv("data/service_requests_clean_v1.csv")

summary = (clean.groupby("region")
                .agg(requests=("request_id", "count"),
                     avg_days=("days_to_resolve", "mean"),
                     avg_satisfaction=("satisfaction_rating", "mean"),
                     total_fees=("processing_fee", "sum"))
                .round(2)
                .sort_values("requests", ascending=False))
summary

,requests,avg_days,avg_satisfaction,total_fees
region,,,,
REGION IV-A,218,7.76,2.95,15000.0
REGION VI,212,6.17,3.03,13350.0
REGION III,201,6.53,2.90,12300.0
NCR,196,6.14,2.98,14050.0
REGION XI,196,6.45,3.06,12850.0
REGION VII,177,5.68,2.91,10900.0


### Hands-On 1 — Grouped Summaries

1. `by_service` — per `service_type`: request count, mean days to resolve, and mean satisfaction. Sort by count descending.
2. `by_office` — per `office_code`: request count and median days to resolve.
3. `two_level` — grouped by `region` **and** `status`, showing request counts. Reset the index so it is a flat table.
4. `busiest_pair` — the region and status combination with the most requests.

In [ ]:
import pandas as pd

# Load the cleaned dataset
clean = pd.read_csv("data/service_requests_clean_v1.csv")

# ============================================================
# 1. by_service
# Per service_type:
# - request count
# - mean days to resolve
# - mean satisfaction
# Sort by request count from highest to lowest
# ============================================================

by_service = (
    clean.groupby("service_type")
    .agg(
        request_count=("request_id", "count"),
        mean_days_to_resolve=("days_to_resolve", "mean"),
        mean_satisfaction=("satisfaction_rating", "mean")
    )
    .sort_values("request_count", ascending=False)
)

# ============================================================
# 2. by_office
# Per office_code:
# - request count
# - median days to resolve
# ============================================================

by_office = (
    clean.groupby("office_code")
    .agg(
        request_count=("request_id", "count"),
        median_days_to_resolve=("days_to_resolve", "median")
    )
)

# ============================================================
# 3. two_level
# Group by BOTH region and status
# Show the number of requests for each combination
# reset_index() converts the grouped index back into columns
# ============================================================

two_level = (
    clean.groupby(["region", "status"])
    .agg(
        request_count=("request_id", "count")
    )
    .reset_index()
)

# ============================================================
# 4. busiest_pair
# Find the region + status combination with the most requests
# ============================================================

busiest_pair = (
    two_level
    .sort_values("request_count", ascending=False)
    .iloc[0]
)

# ============================================================
# Display results
# ============================================================

print(by_service)
print()
print(by_office)
print()
print(two_level.head())
print()
print("Busiest:", busiest_pair)

                            request_count  mean_days_to_resolve  \
service_type                                                      
Free WiFi Installation                238              6.026144   
eGov App Support                      200              6.251852   
Business Permit Assistance            198              6.579710   
Digital Literacy Training             197              6.651852   
ICT Equipment Request                 192              6.861538   
Cybersecurity Advisory                175              6.745455   

                            mean_satisfaction  
service_type                                   
Free WiFi Installation               2.985714  
eGov App Support                     2.867769  
Business Permit Assistance           3.093023  
Digital Literacy Training            3.064000  
ICT Equipment Request                2.837398  
Cybersecurity Advisory               2.980392  

             request_count  median_days_to_resolve
office_code                

## Analysis and Discussion 1

1. **Which service type has the worst combination of high volume and slow resolution? What would you do about it?**

   The service type with the highest request volume and relatively slow average resolution should be identified by comparing both `request_count` and `mean_days_to_resolve` in `by_service`. Rather than looking only at the slowest service or the busiest service, focus on the service that is **both frequently requested and relatively slow to resolve**.

   I would investigate the causes of delays, such as workload, staffing, process bottlenecks, or resource requirements, and then prioritize process improvements for that service.

2. **`by_office` uses median rather than mean. Why is that the better choice here?**

   The **median** is less affected by unusually high or low values than the mean. `days_to_resolve` can contain extreme values or skewed observations, so the median gives a better representation of the **typical resolution time** for an office.

   This is especially useful when a small number of unusually long cases could pull the mean upward.

3. **Grouping by two columns produced many rows. When does a two-level grouping become too detailed to be useful?**

   A two-level grouping becomes too detailed when the number of groups becomes large and many groups contain only a few observations. This can make the results difficult to interpret and may hide the overall patterns.

   For example, combining `region`, `status`, `service_type`, and `channel` would create many small groups. In that situation, it may be better to summarize at a higher level or focus only on groupings that directly answer the analysis question.

## Lecture 2 — Pivot Tables

A pivot table is a grouped summary laid out in two dimensions.

```
pd.pivot_table(df, values="days_to_resolve",
               index="region", columns="service_type",
               aggfunc="mean", fill_value=0)
```

- `index` becomes the rows.
- `columns` becomes the columns.
- `values` is what gets summarized.
- `aggfunc` is how: mean, sum, count, min, max.
- `fill_value` replaces the empty cells that appear where a combination has no data.

**Percentage of total** is where pivots earn their keep. Divide by the row total for share-within-region, or by the grand total for share-of-everything. `margins=True` adds the totals row and column.

In [ ]:
import pandas as pd
clean = pd.read_csv("data/service_requests_clean_v1.csv")

pivot = pd.pivot_table(clean, values="request_id", index="region",
                       columns="service_type", aggfunc="count", fill_value=0)
print("Counts:")
print(pivot)

pct = pivot.div(pivot.sum(axis=1), axis=0).mul(100).round(1)
print("\nPercentage of each region's requests:")
print(pct)

Counts:
service_type  Business Permit Assistance  Cybersecurity Advisory  \
region                                                             
NCR                                   28                      29   
REGION III                            34                      28   
REGION IV-A                           32                      32   
REGION VI                             43                      29   
REGION VII                            34                      28   
REGION XI                             27                      29   

service_type  Digital Literacy Training  Free WiFi Installation  \
region                                                            
NCR                                  28                      49   
REGION III                           25                      36   
REGION IV-A                          46                      46   
REGION VI                            35                      43   
REGION VII                           31      

### Hands-On 2 — Pivot and Percentage

1. `pivot_counts` — request counts, rows = region, columns = channel, `fill_value=0`.
2. `pivot_pct` — the same table as a percentage of each region's total, rounded to 1 decimal. Each row should sum to 100.
3. `pivot_days` — mean days to resolve, rows = region, columns = status, rounded to 1 decimal.
4. `with_totals` — `pivot_counts` rebuilt with `margins=True` and `margins_name="Total"`.

In [ ]:
import pandas as pd

# ============================================================
# SETUP — Load the cleaned dataset
# ============================================================

clean = pd.read_csv("data/service_requests_clean_v1.csv")

# ============================================================
# 1. pivot_counts — Request counts by region and channel
# ============================================================

# Rows = region
# Columns = channel
# Values = request_id
# Count = number of requests
# Empty combinations are replaced with 0
pivot_counts = pd.pivot_table(
    clean,
    values="request_id",
    index="region",
    columns="channel",
    aggfunc="count",
    fill_value=0
)

# ============================================================
# 2. pivot_pct — Percentage of each region's total
# ============================================================

# Divide each value by the total for its region (row).
# Multiply by 100 to convert to percentage.
# Round to 1 decimal place.
pivot_pct = (
    pivot_counts
    .div(pivot_counts.sum(axis=1), axis=0)
    .mul(100)
    .round(1)
)

# ============================================================
# 3. pivot_days — Mean days to resolve by region and status
# ============================================================

# Rows = region
# Columns = status
# Values = days_to_resolve
# Aggregation = mean
# Round the results to 1 decimal place
pivot_days = pd.pivot_table(
    clean,
    values="days_to_resolve",
    index="region",
    columns="status",
    aggfunc="mean"
).round(1)

# ============================================================
# 4. with_totals — Request counts with row and column totals
# ============================================================

# Rebuild pivot_counts with margins=True.
# margins_name="Total" changes the default "All" label to "Total".
with_totals = pd.pivot_table(
    clean,
    values="request_id",
    index="region",
    columns="channel",
    aggfunc="count",
    fill_value=0,
    margins=True,
    margins_name="Total"
)

# ============================================================
# Display all results
# ============================================================

print("Pivot Counts:")
print(pivot_counts)

print("\nPivot Percentages:")
print(pivot_pct)

print("\nPivot Mean Days to Resolve:")
print(pivot_days)

print("\nPivot Counts with Totals:")
print(with_totals)

Pivot Counts:
channel      Email  Hotline  Mobile App  Online Portal  Walk-in
region                                                         
NCR             26       33          34             41       62
REGION III      37       29          38             38       59
REGION IV-A     35       35          26             51       71
REGION VI       25       37          29             61       60
REGION VII      24       26          21             47       59
REGION XI       32       30          27             47       60

Pivot Percentages:
channel      Email  Hotline  Mobile App  Online Portal  Walk-in
region                                                         
NCR           13.3     16.8        17.3           20.9     31.6
REGION III    18.4     14.4        18.9           18.9     29.4
REGION IV-A   16.1     16.1        11.9           23.4     32.6
REGION VI     11.8     17.5        13.7           28.8     28.3
REGION VII    13.6     14.7        11.9           26.6     33.3
REGION

## Analysis and Discussion 2

1. **In `pivot_days`, the Pending column shows 0. Is that true? What should it show instead?**

   No. A `0` does not necessarily mean that pending requests took 0 days to resolve. Pending requests do not have a completed resolution time, so their `days_to_resolve` values are missing (`NaN`). The pivot table should therefore show **`NaN` or a blank cell**, rather than 0, when there are no valid resolution-day values for pending requests.

2. **The percentage view and the count view lead to different conclusions about which region relies on walk-ins. Which would you present, and to whom?**

   The **count view** is useful for managers who need to know the actual workload handled through walk-ins because it shows the number of requests.

   The **percentage view** is more useful when comparing regions of different sizes because it shows the proportion of each region's requests that came through walk-ins.

   I would present the **count view to operations managers** for workload and staffing decisions, while presenting the **percentage view to management or program planners** when comparing channel preferences across regions. Ideally, I would present both because they answer different questions.

3. **When does `fill_value=0` help, and when does it lie?**

   `fill_value=0` helps when an empty combination genuinely means **zero observations**. For example, if a region received no requests through a particular channel, showing 0 makes the table easier to read and calculate.

   However, it can mislead when an empty cell means **missing or unavailable data rather than zero**. For example, in `pivot_days`, replacing a missing mean resolution time with 0 would incorrectly suggest that requests were resolved in zero days. Therefore, `fill_value=0` should be used carefully depending on what an empty cell actually represents.

## Lecture 3 — Frequency Analysis and Binning

**Frequency analysis** counts how often each value occurs. `.value_counts()` does it for one column; `pd.crosstab()` does it for two.

**Binning** converts a continuous number into ordered categories, which makes it usable in frequency analysis and much easier for non-technical audiences to read.

```
pd.cut(df["days"], bins=[0, 3, 10, 30, 999],
       labels=["Same week", "Within 2 weeks", "Within a month", "Over a month"])
```

Two rules for good bins:
1. Choose boundaries that mean something to the audience — a service standard, a policy threshold — not just equal widths.
2. Name the groups clearly. "Bin 3" tells a director nothing; "Over a month" tells him everything.

`pd.qcut()` splits into equal-sized groups instead of equal-width ranges, which is useful for quartile analysis.

In [ ]:
import pandas as pd
clean = pd.read_csv("data/service_requests_clean_v1.csv")

clean["speed_band"] = pd.cut(clean["days_to_resolve"],
                             bins=[-0.1, 3, 10, 30, 1000],
                             labels=["0-3 days", "4-10 days", "11-30 days", "Over 30 days"])
print(clean["speed_band"].value_counts().sort_index())

print("\nSpeed band by channel:")
print(pd.crosstab(clean["channel"], clean["speed_band"]))

speed_band
0-3 days        291
4-10 days       372
11-30 days      129
Over 30 days      9
Name: count, dtype: int64

Speed band by channel:
speed_band     0-3 days  4-10 days  11-30 days  Over 30 days
channel                                                     
Email                39         56          25             2
Hotline              48         53          17             1
Mobile App           52         55          18             0
Online Portal        66         86          31             2
Walk-in              86        122          38             4


### Hands-On 3 — Bin and Cross-Tabulate

1. Add `speed_band` to `clean` using the bins and labels from the demo.
2. `band_counts` — the count per band, in band order.
3. `band_by_region` — a crosstab of region against speed_band.
4. `band_pct` — the same crosstab normalized to row percentages, rounded to 1 decimal (use `normalize="index"`).
5. `worst_region` — the region with the highest share of requests in the "Over 30 days" band.

In [ ]:
import pandas as pd

# Load the cleaned dataset
clean = pd.read_csv("data/service_requests_clean_v1.csv")

# ============================================================
# 1. Add speed_band
# ============================================================

# Convert days_to_resolve into meaningful time categories
clean["speed_band"] = pd.cut(
    clean["days_to_resolve"],
    bins=[-0.1, 3, 10, 30, 1000],
    labels=[
        "0-3 days",
        "4-10 days",
        "11-30 days",
        "Over 30 days"
    ]
)

# ============================================================
# 2. band_counts — Count per speed band
# ============================================================

# Count the number of requests in each speed band.
# sort_index() preserves the logical order of the categories.
band_counts = (
    clean["speed_band"]
    .value_counts()
    .sort_index()
)

# ============================================================
# 3. band_by_region — Crosstab of region against speed_band
# ============================================================

# Rows = region
# Columns = speed_band
# Values = number of requests
band_by_region = pd.crosstab(
    clean["region"],
    clean["speed_band"]
)

# ============================================================
# 4. band_pct — Row percentages
# ============================================================

# normalize="index" calculates the percentage within each row.
# This means each region's speed-band percentages add up to 100%.
# Multiply by 100 and round to one decimal place.
band_pct = (
    pd.crosstab(
        clean["region"],
        clean["speed_band"],
        normalize="index"
    )
    .mul(100)
    .round(1)
)

# ============================================================
# 5. worst_region — Highest share of "Over 30 days"
# ============================================================

# Find the region with the highest percentage of requests
# in the "Over 30 days" category.
worst_region = band_pct["Over 30 days"].idxmax()

# ============================================================
# Display results
# ============================================================

print(band_counts)
print()
print(band_by_region)
print()
print(band_pct)
print()
print("Worst:", worst_region)

speed_band
0-3 days        291
4-10 days       372
11-30 days      129
Over 30 days      9
Name: count, dtype: int64

speed_band   0-3 days  4-10 days  11-30 days  Over 30 days
region                                                    
NCR                58         57          19             2
REGION III         42         75          19             1
REGION IV-A        42         74          28             5
REGION VI          51         60          23             0
REGION VII         45         52          17             0
REGION XI          53         54          23             1

speed_band   0-3 days  4-10 days  11-30 days  Over 30 days
region                                                    
NCR              42.6       41.9        14.0           1.5
REGION III       30.7       54.7        13.9           0.7
REGION IV-A      28.2       49.7        18.8           3.4
REGION VI        38.1       44.8        17.2           0.0
REGION VII       39.5       45.6        14.9           

## Analysis and Discussion 3

1. **Would your bin boundaries change if DICT published a 7-day service standard? Redesign them.**

   Yes. If a 7-day service standard were published, the bin boundaries should reflect that standard instead of using the original 3-day, 10-day, and 30-day thresholds.

   For example, the bins could be redesigned as:

   - `0-7 days` — within the service standard
   - `8-14 days` — up to one week beyond the standard
   - `15-30 days` — significantly beyond the standard
   - `Over 30 days` — substantially delayed

   This makes the categories more meaningful because the first boundary directly represents the **7-day service standard**.

2. **The count crosstab and the percentage crosstab identify different "worst" regions. Why, and which is fairer?**

   They can identify different regions because they measure different things. The **count crosstab** shows the actual number of requests in each speed band, while the **percentage crosstab** shows the proportion of a region's requests that fall into each speed band.

   A larger region may have more delayed requests simply because it receives more requests overall. A smaller region may have fewer delayed requests in absolute numbers but a much higher percentage of its requests may be delayed.

   For comparing regions with different numbers of requests, the **percentage view is generally fairer** because it accounts for differences in workload. However, the count view is still important when the objective is to understand the actual volume of delayed cases and resource requirements.

3. **What does binning throw away? When would you refuse to bin and show the raw distribution instead?**

   Binning throws away some of the precise information in the original numeric values. For example, requests taking 11 days and 29 days are both placed in the `11-30 days` category even though their actual resolution times are different.

   I would avoid binning when the exact values are important, when small differences matter, or when the distribution itself needs to be examined in detail. For example, when analyzing the shape of `days_to_resolve`, identifying extreme values, or performing statistical analysis, I would show the **raw distribution** using a histogram, boxplot, or summary statistics instead.